## AutoFunctions Testing

In [ ]:
%load_ext autoreload
%autoreload 2


from fibsem.autofunctions.autofocus import run_auto_focus, AutoFocusResult, AutoFocusIteration, AutoFocusSettings, FocusSweepPass
from fibsem.autofunctions.plotting import plot_autofocus_result
from fibsem import utils, acquire
from fibsem.structures import ImageSettings, BeamType

microscope, settings = utils.setup_session()




In [ ]:
image = microscope.acquire_image(None, BeamType.ION)
wd = microscope.get_working_distance(BeamType.ION)



In [ ]:
wd

In [ ]:
from fibsem.structures import FibsemRectangle

import random

perturb = random.random() * 500e-6
print(perturb)
if random.random() > 0.5:
    perturb*= -1

# break
microscope.set_working_distance(wd+perturb, BeamType.ION)

reduced_area = FibsemRectangle(0.25, 0.25, 0.5, 0.5)
# microscope.autocontrast(BeamType.ION, reduced_area=reduced_area)


# query: if focus is too far from eucentric, do a sanity check to move it back before starting? -> side effects

result = run_auto_focus(
    microscope,
    beam_type=BeamType.ION, hfw=100e-6,
    settings=AutoFocusSettings(
        method="tenengrad",
        passes=[
            FocusSweepPass(n_steps=10, step_size=100e-6),
            FocusSweepPass(n_steps=10, step_size=10e-6),
        ],
        reduced_area=reduced_area,
        use_autocontrast=True)
    )

image = microscope.acquire_image(None, BeamType.ION)



In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import os
path = os.path.join(os.getcwd(), "autofunctions")
os.makedirs(path, exist_ok=True)
fig = plot_autofocus_result(result, save_path=os.path.join(path, "autofocus_result.png"))

#show figure
plt.figure(1)
plt.show()

In [ ]:
fig

In [ ]:
from fibsem.autofunctions.acb import run_auto_contrast_brightness, AutoContrastBrightnessSettings
from fibsem.autofunctions.plotting import plot_acb_result

beam_type = BeamType.ELECTRON
microscope.set_detector_brightness(0.77, beam_type)
microscope.set_detector_contrast(0.66, beam_type)

settings = AutoContrastBrightnessSettings(
    n_iterations=15,
)

result = run_auto_contrast_brightness(
    microscope=microscope,
    beam_type=beam_type,
    hfw=100e-6,
    settings=settings,
)


fig = plot_acb_result(result, save_path=os.path.join(path, "acb_result.png"))

In [ ]:
plt.show()
fig
